In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

env_path = Path(r"D:\arun\ai-learning\.env")

load_dotenv(env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded:",bool(api_key))

API key loaded: True


In [6]:
import chromadb
from sentence_transformers import SentenceTransformer

In [7]:
CHROMA_PATH = "../ingestion/chroma_db"

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

print("Chroma connected successfully")

Chroma connected successfully


In [8]:
CHROMA_PATH = "../ingestion/chroma_db"

client = chromadb.PersistentClient(
    path=CHROMA_PATH
)

In [9]:
collection = client.get_collection(
    name="knowledge_base"
)

print("Stored chunks:", collection.count())

Stored chunks: 147


In [10]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
def retrieve(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query]
    )

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=top_k
    )

    retrieved_chunks = []

    for i in range(len(results["documents"][0])):
        retrieved_chunks.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved_chunks

In [12]:
query = "What is Python?"

results = retrieve(query, top_k=3)

print("Retrieved:", len(results))

Retrieved: 3


In [13]:
for i, result in enumerate(results, start=1):
    print("=" * 80)
    print("RESULT:", i)
    print("DISTANCE:", result["distance"])
    print("METADATA:", result["metadata"])
    print("TEXT:")
    print(result["text"][:500])

RESULT: 1
DISTANCE: 1.7633297443389893
METADATA: {'chunk_id': 3, 'source': 'long-doc.txt'}
TEXT:
elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetu
RESULT: 2
DISTANCE: 1.7633297443389893
METADATA: {'source': 'long-doc.txt', 'chunk_id': 7}
TEXT:
elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum do

In [14]:
results = retrieve(query, top_k=5)

print("Retrieved:", len(results))

Retrieved: 5


In [15]:
def build_rag_prompt(query, retrieved_chunks):

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):

        source = chunk["metadata"].get(
            "source",
            "Unknown"
        )

        page = chunk["metadata"].get(
            "page",
            "N/A"
        )

        context_parts.append(
            f"[Source {i}: {source}, page {page}]\n"
            f"{chunk['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a helpful RAG assistant.

Answer the question using ONLY the provided context.

If the answer is not present in the context,
say that the information is not available in
the provided documents.

Do not invent facts.

CONTEXT:
----------------
{context}
----------------

QUESTION:
{query}

Provide a concise answer and mention the relevant sources.
"""

    return prompt

In [16]:
query = "What is Python?"

retrieved = retrieve(
    query,
    top_k=3
)

prompt = build_rag_prompt(
    query,
    retrieved
)

print(prompt)


You are a helpful RAG assistant.

Answer the question using ONLY the provided context.

If the answer is not present in the context,
say that the information is not available in
the provided documents.

Do not invent facts.

CONTEXT:
----------------
[Source 1: long-doc.txt, page N/A]
elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit amet, consectetur adipiscing elit. Lorem ipsum dolor sit a

In [21]:
from openai import OpenAI
import os

In [22]:
client_llm = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [29]:
def generate(prompt):

    model = "openai/gpt-oss-20b"
    response = client_llm.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer only from the supplied context. "
                    "Do not invent information."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [37]:
query = "What are the key features of the SmartHome Hub?"

retrieved = retrieve(
    query,
    top_k=5
)

prompt = build_rag_prompt(
    query,
    retrieved
)

answer = generate(prompt)

print(answer)

The SmartHome Hub is a central device that **connects and controls all smart‑home devices seamlessly**.  
*Source: Sample‑text‑only‑pdf‑a4‑size.pdf, page 1*


In [38]:
def get_sources(retrieved_chunks):

    sources = []

    for chunk in retrieved_chunks:

        metadata = chunk["metadata"]

        source = metadata.get(
            "source",
            "Unknown"
        )

        page = metadata.get(
            "page",
            "N/A"
        )

        sources.append(
            f"{source}, page {page}"
        )

    return list(dict.fromkeys(sources))

In [39]:
sources = get_sources(retrieved)

print("ANSWER")
print(answer)

print("\nSOURCES")
for source in sources:
    print("-", source)

ANSWER
The SmartHome Hub is a central device that **connects and controls all smart‑home devices seamlessly**.  
*Source: Sample‑text‑only‑pdf‑a4‑size.pdf, page 1*

SOURCES
- sample-text-only-pdf-a4-size.pdf, page 4
- sample-text-only-pdf-a4-size.pdf, page 0
- sample-text-only-pdf-a4-size.pdf, page 1
- sample-pdf-a4-size-65kb.pdf, page 2
- basic-text.pdf, page 0


In [46]:
def generate_without_retrieval(query):

    model = "openai/gpt-oss-20b"
    response = client_llm.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": query
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [47]:
query = "What does the provided document say about SmartHome Hub?"

without_rag = generate_without_retrieval(query)

print(without_rag)

I’m not able to see the document you’re referring to. Could you paste the relevant portion or give me a brief summary of what it contains? That way I can give you an accurate answer about what it says regarding the SmartHome Hub.


In [48]:
retrieved = retrieve(
    query,
    top_k=5
)

prompt = build_rag_prompt(
    query,
    retrieved
)

with_rag = generate(prompt)

print(with_rag)

The documents describe the **SmartHome Hub** as a new central device that will connect and control all smart‑home appliances, with the goal of revolutionizing home automation and positioning Innovative Tech Solutions as a market leader.  
Key points include:

* It is the focus of a product‑launch report prepared by Alex Johnson (Source 2).  
* The launch strategy outlines objectives such as introducing the hub and its features, presenting market research and competitor analysis, and detailing marketing and sales plans (Source 3).  
* The conclusion highlights the hub’s significant market opportunity, unique features, and a strategic marketing plan, and lists next steps for production, pre‑orders, and the launch event (Source 1).  
* Market data cited in the report shows a $50 billion market size, 85 % user satisfaction, and a 10 % growth rate (Source 5).


In [49]:
print("=" * 80)
print("WITHOUT RETRIEVAL")
print("=" * 80)
print(without_rag)

print("\n")

print("=" * 80)
print("WITH RETRIEVAL")
print("=" * 80)
print(with_rag)

print("\n")

print("=" * 80)
print("SOURCES")
print("=" * 80)

for source in get_sources(retrieved):
    print("-", source)

WITHOUT RETRIEVAL
I’m not able to see the document you’re referring to. Could you paste the relevant portion or give me a brief summary of what it contains? That way I can give you an accurate answer about what it says regarding the SmartHome Hub.


WITH RETRIEVAL
The documents describe the **SmartHome Hub** as a new central device that will connect and control all smart‑home appliances, with the goal of revolutionizing home automation and positioning Innovative Tech Solutions as a market leader.  
Key points include:

* It is the focus of a product‑launch report prepared by Alex Johnson (Source 2).  
* The launch strategy outlines objectives such as introducing the hub and its features, presenting market research and competitor analysis, and detailing marketing and sales plans (Source 3).  
* The conclusion highlights the hub’s significant market opportunity, unique features, and a strategic marketing plan, and lists next steps for production, pre‑orders, and the launch event (Source 

In [50]:
for result in retrieved:
    print(
        result["metadata"],
        "distance =",
        result["distance"]
    )

{'creationdate': '2024-07-07T12:12:22+00:00', 'chunk_id': 146, 'page_label': '5', 'title': 'Sample Text Only PDF A4 Size | Sample-Files.com', 'total_pages': 5, 'source': 'sample-text-only-pdf-a4-size.pdf', 'creator': 'Canva', 'page': 4, 'keywords': 'DAGKQ_EHWeE,BAEVm-WErFA', 'moddate': '2024-07-07T12:12:21+00:00', 'producer': 'Canva', 'author': 'Jericho'} distance = 0.45175856351852417
{'author': 'Jericho', 'total_pages': 5, 'page': 0, 'creationdate': '2024-07-07T12:12:22+00:00', 'creator': 'Canva', 'producer': 'Canva', 'source': 'sample-text-only-pdf-a4-size.pdf', 'chunk_id': 142, 'keywords': 'DAGKQ_EHWeE,BAEVm-WErFA', 'moddate': '2024-07-07T12:12:21+00:00', 'title': 'Sample Text Only PDF A4 Size | Sample-Files.com', 'page_label': '1'} distance = 0.45994842052459717
{'page_label': '2', 'chunk_id': 143, 'total_pages': 5, 'keywords': 'DAGKQ_EHWeE,BAEVm-WErFA', 'source': 'sample-text-only-pdf-a4-size.pdf', 'page': 1, 'creator': 'Canva', 'moddate': '2024-07-07T12:12:21+00:00', 'producer':